# Agricultural Crop Monitoring
**NB 05** | Nile Delta, Egypt | Sentinel-2 time-series | NDVI + SegFormer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)


In [ ]:
BBOX       = (30.8, 30.3, 31.2, 30.7)
DATE_RANGE = ('2024-03-01', '2024-10-31')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 20,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B04', 'B08', 'B11']

downloads = client.download(
    results[:6],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32636', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 3,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'nile_delta_preprocessed.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'segmentation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# NDVI time-series analysis on preprocessed data
if PREPROCESSED.exists():
    idx_dir = DATA_DIR / 'ndvi_series'
    idx_dir.mkdir(exist_ok=True)
    # Band map for BANDS=[B04, B08, B11]: red=1, nir=2, swir1=3
    ndvi_path = client.indices.ndvi(str(PREPROCESSED), red_band=1, nir_band=2,
        output_path=str(idx_dir/'ndvi.tif'))
    import rasterio
    with rasterio.open(ndvi_path) as src:
        ndvi_arr = src.read(1)
    print(f"NDVI statistics:")
    print(f"  Mean NDVI : {ndvi_arr[ndvi_arr > -1].mean():.3f}")
    print(f"  Max NDVI  : {ndvi_arr.max():.3f}")
    print(f"  Veg cover : {(ndvi_arr > 0.3).mean()*100:.1f}% (NDVI > 0.3)")
    print(f"  Crop area : {(ndvi_arr > 0.4).mean()*100:.1f}% (NDVI > 0.4)")


In [ ]:
PREDICTION = DATA_DIR / 'crop_map.tif'

if ready and PREPROCESSED.exists():
    model = get_model('segformer-b2', num_classes=3, pretrained=True)
    inf   = TiledInference(model, chip_size=512, overlap=64, blend_mode='gaussian')
    res   = inf.infer(str(PREPROCESSED), str(PREDICTION))
    print(f"Inference: {res.get('n_chips')} chips  {res.get('duration_seconds',0):.1f}s  "
          f"{res.get('chips_per_second',0):.1f} chips/s")
else:
    print("Skipping inference — no preprocessed scene")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=10, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=100.0, simplify_tolerance=1.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


In [ ]:
# Zonal statistics per field polygon
VECTOR  = DATA_DIR / 'output.geojson'
if VECTOR.exists():
    idx_dir = DATA_DIR / 'ndvi_series'
    ndvi_path = idx_dir / 'ndvi.tif'
    if ndvi_path.exists():
        zonal = client.postprocess.zonal_statistics(
            str(ndvi_path), str(VECTOR),
            output_path=str(DATA_DIR/'crop_ndvi_stats.geojson'),
            stats=['mean','std','min','max','count'],
        )
        print(f"Zonal statistics saved: {zonal}")


**NB05 — Crop Monitoring** complete.
- Study area: Nile Delta Egypt
- Sensor: Sentinel-2 seasonal
- Model: SegFormer-B2 + NDVI
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG